## Mapping 

In [2]:
import torch
import torch.nn as nn

# 1. 고정된 고차원 입력 데이터 (4x1 행렬 / 4차원 벡터)
# 예: [주택크기, 방개수, 범죄율, 학군점수]
x = torch.tensor([[2.0], 
                  [3.0], 
                  [0.1], 
                  [9.0]], dtype=torch.float32)

# 2. 4차원을 2차원으로 매핑하는 변환 행렬 A (2x4 행렬)
# 실무에서는 nn.Linear가 이 행렬(가중치)을 내포하고 있습니다.
A = torch.tensor([[0.5,  1.2, -0.3,  0.1],
                  [0.1, -0.5,  0.8,  1.5]], dtype=torch.float32)

# 3. 행렬 곱을 통한 사상 (Mapping) 수행: Y = A * X
y = torch.matmul(A, x)

print("=== 4차원 입력 벡터 x ===")
print(x)
print("Shape:", x.shape) # Output: torch.Size([4, 1])

print("\n=== 변환 행렬 (사상 함수) A ===")
print(A)
print("Shape:", A.shape) # Output: torch.Size([2, 4])

print("\n=== 2차원 평면으로 매핑된 결과 벡터 y ===")
print(y)
print("Shape:", y.shape) # Output: torch.Size([2, 1])

=== 4차원 입력 벡터 x ===
tensor([[2.0000],
        [3.0000],
        [0.1000],
        [9.0000]])
Shape: torch.Size([4, 1])

=== 변환 행렬 (사상 함수) A ===
tensor([[ 0.5000,  1.2000, -0.3000,  0.1000],
        [ 0.1000, -0.5000,  0.8000,  1.5000]])
Shape: torch.Size([2, 4])

=== 2차원 평면으로 매핑된 결과 벡터 y ===
tensor([[ 5.4700],
        [12.2800]])
Shape: torch.Size([2, 1])


## 1. 회전 행렬이란 무엇인가? (정의)

**회전 행렬**은 2차원 평면(혹은 3차원 공간)에 있는 어떤 점이나 벡터를, 원점을 기준으로 특정 각도 $\theta$만큼 회전시키는 '선형 변환 행렬'입니다. 벡터의 길이나 모양은 전혀 찌그러뜨리지 않고 오직 '방향'만 바꾸는 매우 특별한 포탈입니다.

2차원 평면($\mathbb{R}^2$)에서 각도 $\theta$만큼 반시계 방향으로 회전시키는 행렬 $R(\theta)$는 다음과 같이 생겼습니다.

$$R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$$

만약 우리가 어떤 2차원 벡터 $v = \begin{pmatrix} x \ y \end{pmatrix}$를 $\theta$만큼 회전시켜 새로운 벡터 $v'$를 만들고 싶다면, 단순히 이 행렬을 곱해주면 됩니다.

$$v' = R(\theta)v = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix} = \begin{pmatrix} x\cos\theta - y\sin\theta \\ x\sin\theta + y\cos\theta \end{pmatrix}$$

---

## 2. 직관적인 증명: 왜 저런 모양일까?

고등학교 수학에서는 삼각함수의 덧셈정리를 이용해 이를 증명하지만, 우리는 AI 선구자답게 **'선형대수학의 기저 벡터(Basis Vector)'** 관점에서 아주 쉽고 우아하게 증명해 보겠습니다.

앞서 행렬의 열(Column)은 "원래 축이 변환 후 어디로 이동하는지 보여주는 목적지"라고 설명해 드렸습니다. 2차원 평면을 구성하는 두 개의 기본 축을 살펴봅시다.

* **$x$축의 기본 단위 벡터:** $\mathbf{e}_1 = \begin{pmatrix} 1 \\ 0 \end{pmatrix}$ (동쪽으로 1만큼)
* **$y$축의 기본 단위 벡터:** $\mathbf{e}_2 = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$ (북쪽으로 1만큼)

이 두 축을 원점을 중심으로 반시계 방향으로 $\theta$만큼 회전시키면 어디로 갈까요? 삼각비의 정의(반지름이 1인 단위원 위에서의 좌표)를 떠올려 보세요.

**Step 1: $x$축 벡터의 이동**
$\begin{pmatrix} 1 \ 0 \end{pmatrix}$을 $\theta$만큼 회전시키면, 단위원 위의 점이 되므로 그 좌표는 자연스럽게 $\begin{pmatrix} \cos\theta \ \sin\theta \end{pmatrix}$가 됩니다.
👉 이것이 회전 행렬의 첫 번째 열(Column)이 됩니다!

**Step 2: $y$축 벡터의 이동**
$\begin{pmatrix} 0 \\ 1 \end{pmatrix}$을 $\theta$만큼 회전시켜 봅시다. 이 벡터는 항상 $x$축 벡터보다 $90^\circ$ (${\pi \over 2}$) 앞서 있습니다.
$\cos(\theta + 90^\circ) = -\sin\theta$
$\sin(\theta + 90^\circ) = \cos\theta$
따라서 이 벡터가 도착하는 곳은 $\begin{pmatrix} -\sin\theta \ \cos\theta \end{pmatrix}$입니다.
👉 이것이 회전 행렬의 두 번째 열(Column)이 됩니다!

**Step 3: 행렬 완성**
변환 행렬은 단순히 변환된 기저 벡터들을 열로 나열한 것입니다.


$$R(\theta) = \begin{pmatrix} (\mathbf{e}_1\text{이 도착한 x}) & (\mathbf{e}_2\text{가 도착한 x}) \\ (\mathbf{e}_1\text{이 도착한 y}) & (\mathbf{e}_2\text{가 도착한 y}) \end{pmatrix} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$$

복잡한 계산 없이, 오직 $x$축과 $y$축 막대기 두 개가 단위원 위에서 어디로 돌아가는지만 관찰하면 증명이 끝납니다. 참 아름답지 않나요?

---

## 3. 현대 AI 및 기술에서의 활용

회전 행렬은 고차원 공간의 방향성을 다루는 모든 분야에서 필수적입니다.

* **컴퓨터 비전 (Computer Vision):** 자율주행 자동차가 카메라로 표지판을 볼 때, 표지판이 기울어져 있어도(Rotation), 이 회전 행렬의 역행렬을 곱해 정면에서 본 것처럼 이미지를 보정(Alignment)합니다.
* **로보틱스와 강화학습 (Robotics & RL):** 로봇 팔의 관절이 움직일 때, 각 관절의 회전 각도를 계산하여 로봇 손끝의 3차원 최종 위치를 찾아내는 순운동학(Forward Kinematics) 계산의 핵심입니다.
* **3D 그래픽스 & 메타버스:** 가상 공간에서 카메라의 시점을 돌리거나 캐릭터가 고개를 돌릴 때 매 프레임마다 이 연산이 GPU에서 수행됩니다.


In [3]:
# Affine Transformation (선형 변환 + 편향)
import torch
import torch.nn as nn

# 2차원 데이터를 입력받아 3차원으로 사상(Mapping)하는 아핀 변환 레이어 생성
# in_features=2, out_features=3
affine_layer = nn.Linear(in_features=2, out_features=3, bias=True)

# 1. 임의의 2차원 데이터 포인트 x
x = torch.tensor([1.0, 2.0])

# 2. PyTorch가 내부적으로 연산하는 방식 (y = Wx + b)
y = affine_layer(x)

print("=== AI가 학습한 선형 변환 행렬 (Weight, W) ===")
print(affine_layer.weight) # 공간의 형태를 변형

print("\n=== AI가 학습한 이동 벡터 (Bias, b) ===")
print(affine_layer.bias)   # 원점에서 벗어나게 해주는 이동(Translation) 역할

print("\n=== 최종 아핀 변환 결과 (y) ===")
print(y)

=== AI가 학습한 선형 변환 행렬 (Weight, W) ===
Parameter containing:
tensor([[-0.4417,  0.0591],
        [-0.4322, -0.0778],
        [-0.2498,  0.0236]], requires_grad=True)

=== AI가 학습한 이동 벡터 (Bias, b) ===
Parameter containing:
tensor([-0.0719, -0.1443, -0.4191], requires_grad=True)

=== 최종 아핀 변환 결과 (y) ===
tensor([-0.3955, -0.7321, -0.6217], grad_fn=<ViewBackward0>)


# 전치 

In [6]:
import numpy as np

a = np.array([[1,2,3], [4,5,6],[7,8,9]])
print(a)
print(a.transpose())
print(a.T)

[[1 2 3]
 [4 5 6]
 [7 8 9]]
[[1 4 7]
 [2 5 8]
 [3 6 9]]
[[1 4 7]
 [2 5 8]
 [3 6 9]]


# 대각합 

**대각합(Trace)**은 **정방행렬(행과 열의 개수가 같은 행렬)**에서 **주대각선(왼쪽 위에서 오른쪽 아래로 이어지는 대각선)에 위치한 모든 원소들을 더한 값**을 의미합니다. 보통 기호로는 $\text{tr}(A)$ 또는 $\text{Tr}(A)$로 표기합니다.

### 1. 수학적 정의
$n \times n$ 정방행렬 $A$가 있을 때, 행렬의 $i$행 $i$열 원소를 $a_{ii}$라고 하면 대각합은 다음과 같이 정의됩니다.

$$ \text{tr}(A) = \sum_{i=1}^{n} a_{ii} = a_{11} + a_{22} + \dots + a_{nn} $$

### 2. 계산 예제 (수학)
현재 작업 중이신 노트북에 있는 3x3 행렬 예시를 살펴봅시다.

$$
A = \begin{pmatrix} 
\mathbf{1} & 2 & 3 \\ 
4 & \mathbf{5} & 6 \\ 
7 & 8 & \mathbf{9}
\end{pmatrix}
$$

이 행렬에서 주대각선 원소는 $1, 5, 9$입니다. 따라서 행렬 $A$의 대각합은 다음과 같이 계산됩니다.

$$ \text{tr}(A) = 1 + 5 + 9 = 15 $$

### 3. 주요 성질
대각합은 물리나 머신러닝 수학에서 굉장히 중요한 특성을 가지는데, 대표적으로 다음과 같은 성질들이 있습니다.
* **불변성**: 자기 자신의 전치행렬과 대각합이 동일합니다. $\text{tr}(A) = \text{tr}(A^T)$
* **선형성**: $\text{tr}(A + B) = \text{tr}(A) + \text{tr}(B)$, $\text{tr}(cA) = c\text{tr}(A)$ (단, $c$는 스칼라 상수)
* **고윳값과의 관계(중요)**: 어떤 행렬의 대각합은 그 행렬이 가지는 **모든 고윳값(Eigenvalues)들의 합과 정확히 일치**합니다.

In [13]:
import numpy as np
import torch

# 1. NumPy를 이용한 대각합 계산
a_np = np.array([[1, 2, 3], 
                 [4, 5, 6], 
                 [7, 8, 9]])

trace_np = np.trace(a_np)
print(f"NumPy 행렬 대각합: np.trace(a_np) = {trace_np}")  # 출력: 15

# 2. PyTorch를 이용한 대각합 계산
a_tensor = torch.tensor([[1, 2, 3], 
                         [4, 5, 6], 
                         [7, 8, 9]])

trace_torch = torch.trace(a_tensor)
print(f"PyTorch 텐서 대각합: torch.trace(a_tensor) = {trace_torch.item()}")  # 출력: 15

print(f"NumPy 대각합과 PyTorch 대각합이 동일한가? {'Yes' if trace_np == trace_torch.item() else 'No'}" )

print(f"np.diag {np.diag(a_np)}") # 대각 성분을 추출하는 함수

NumPy 행렬 대각합: np.trace(a_np) = 15
PyTorch 텐서 대각합: torch.trace(a_tensor) = 15
NumPy 대각합과 PyTorch 대각합이 동일한가? Yes
np.diag [1 5 9]


## 행렬의 거듭 제곱 

- np.dot 을 여러번 호출 
- `matrix_power`함수 사용 => `np.dot`보다 빠름 

In [15]:
from numpy.linalg import matrix_power

A = np.array([[1, 2], [3, 4]])
A_squared = matrix_power(A, 2)
print("A^2:")
print(A_squared)

A^2:
[[ 7 10]
 [15 22]]


# 특별한 정방행렬들

In [20]:
print(np.zeros((3, 5))) # 3행 5열의 영행렬 생성
print(np.zeros(1)) # 스칼라 0 생성
print(help(np.zeros))

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
[0.]
Help on built-in function zeros in module numpy:

zeros(shape, dtype=None, order='C', *, device=None, like=None)
    zeros(shape, dtype=None, order='C', *, device=None, like=None)

    Return a new array of given shape and type, filled with zeros.

    Parameters
    ----------
    shape : int or tuple of ints
        Shape of the new array, e.g., ``(2, 3)`` or ``2``.
    dtype : data-type, optional
        The desired data-type for the array, e.g., `numpy.int8`.  Default is
        `numpy.float64`.
    order : {'C', 'F'}, optional, default: 'C'
        Whether to store multi-dimensional data in row-major
        (C-style) or column-major (Fortran-style) order in memory.
    device : str, optional
        The device on which to place the created array. Default: ``None``.
        For Array-API interoperability only, so must be ``"cpu"`` if passed.

        .. versionadded:: 2.0.0
    like : array_like, optional
        Referenc

In [22]:
print(np.ones((3, 3)))

[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]


# 단위 행렬 ( identity matrix ), 항등 행렬

- 대각 성분은 1이고, 나머지는 0

**단위 행렬(Identity Matrix, 또는 항등 행렬)**은 선형대수학에서 숫자 '1'과 같은 역할을 하는 특수한 정방행렬입니다. 보통 기호로 $I$ 또는 $E$로 표기합니다.

### 1. 단위 행렬의 특징
* **형태**: 주대각선(왼쪽 위에서 오른쪽 아래로 이어지는 대각선)의 모든 원소가 **1**이고, 나머지 모든 원소는 **0**입니다.
* **항등성 (가장 중요한 특징)**: 어떤 행렬 $A$에 단위 행렬 $I$를 곱해도 **자기 자신**이 그대로 나옵니다.
  $$ A \times I = I \times A = A $$
* 딥러닝이나 수학에서 행렬의 역행렬을 정의할 때의 기준이 됩니다 ($A \times A^{-1} = I$).

### 2. 만드는 함수 (NumPy & PyTorch)
파이썬 배열 라이브러리들에서는 단위 행렬을 뜻하는 "Identity"의 발음과 유사한 눈(Eye)을 따와서 주로 `eye()` 라는 이름의 함수를 사용합니다. (NumPy는 `identity()` 함수도 제공합니다.)



NumPy의 경우 `np.eye`는 직사각형 행렬(NxM)에서 대각 성분만 1로 만들거나 다른 대각선을 1로 만드는 추가 기능이 있고, `np.identity`는 오직 정방행렬(NxN)만 만들 수 있다는 미세한 차이가 있습니다. 실무에서는 이름이 짧은 `np.eye()`를 훨씬 더 많이 사용합니다.

In [25]:
import numpy as np
import torch

# 1. NumPy로 생성
# np.eye(n): n x n 크기의 단위 행렬 생성
I_np_eye = np.eye(3)
print("--- np.eye(3) ---")
print(I_np_eye)

# np.identity(n)도 완전히 동일한 역할
I_np_ident = np.identity(3)
print("\n--- np.identity(3) ---")
print(I_np_ident)

# 2. PyTorch로 생성
# torch.eye(n): n x n 크기의 단위 텐서 생성
I_torch = torch.eye(3)
print("\n--- torch.eye(3) ---")
print(I_torch)

print("\n--- A @ I  ---")
a = np.array([[1, 2], [3, 4]])
i = np.identity(2)
print( a @ i ) # 단위 행렬과의 곱은 원래 행렬이 그대로 나옴

--- np.eye(3) ---
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

--- np.identity(3) ---
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

--- torch.eye(3) ---
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])

--- A @ I  ---
[[1. 2.]
 [3. 4.]]


# 행렬 식 

- `np.linalg.det` 함수 사용 

In [26]:
a = np.array([[1, 2], [3, 4]])
print(a)
np.linalg.det(a) # 행렬식 계산


[[1 2]
 [3 4]]


np.float64(-2.0000000000000004)

# 역행렬 (Inverse Matrix)

**역행렬(Inverse Matrix)**은 어떤 숫자 $x$에 곱했을 때 1이 되게 만드는 역수($1/x$)와 같은 개념의 행렬입니다. 어떤 정방행렬 $A$에 곱했을 때 **단위 행렬($I$)이 되게 만드는 행렬**을 $A$의 역행렬이라고 하며, 기호로는 $A^{-1}$로 표기합니다.

### 1. 수학적 정의
정방행렬 $A$와 그 역행렬 $A^{-1}$ 사이에는 다음의 관계가 성립합니다.
$$ A \times A^{-1} = A^{-1} \times A = I $$

### 2. 역행렬의 존재 조건 (행렬식)
모든 행렬이 역행렬을 가지는 것은 아닙니다. 
* 역행렬이 존재하려면 행렬이 정사각형(정방행렬)이어야 합니다.
* **행렬식(Determinant)이 0이 아니어야 합니다.** ($\text{det}(A) \neq 0$)
* 행렬식이 0인 행렬은 '특이 행렬(Singular Matrix)'이라고 하며, 역행렬이 존재하지 않습니다.

### 3. 왜 필요한가? (선형 연립방정식 풀이)
기본적인 방정식 $ax = b$를 풀 때 $x = a^{-1}b$ 이듯, 연립방정식을 행렬 형태로 나타낸 $Ax = B$에서 미지수 역터 $x$를 구하기 위해 역행렬을 사용합니다.
$$ x = A^{-1}B $$
딥러닝 및 머신러닝에서는 파라미터(가중치)를 최적화하는 과정이나 행렬 분해 등 수많은 곳에서 이론적 기반으로 사용됩니다.

### 4. 만드는 함수 (NumPy & PyTorch)
역행렬은 `np.linalg.inv()` 함수 또는 `torch.linalg.inv()` 함수를 사용하여 계산할 수 있습니다.

### 5. 역행렬의 주요 특성

#### (1) 두 행렬 곱의 역행렬 (순서가 뒤집힘)
두 정방행렬 $A$와 $B$가 각각 역행렬을 가질 때, 두 행렬을 곱한 값의 역행렬은 **각각의 역행렬을 곱하되 순서가 반대로** 뒤집힙니다. 전치행렬(Transpose)의 성질과 동일합니다.
$$ (AB)^{-1} = B^{-1} A^{-1} $$

**왜 그럴까요?**
$(AB)$에 무엇을 곱해야 단위 행렬 $I$가 나오는지를 생각해보면 알 수 있습니다.
$ (AB)(B^{-1}A^{-1}) = A(BB^{-1})A^{-1} = AIA^{-1} = AA^{-1} = I $

#### (2) 대각 행렬(Diagonal Matrix)의 역행렬
대각 행렬(주대각선 성분을 제외한 모든 원소가 0인 행렬)의 역행렬은 **매우 구하기 쉽습니다**. 
복잡한 행렬식 계산 없이, **주대각선의 각 원소를 각각 역수($1/x$)로** 취해주기만 하면 됩니다. (단, 모든 대각 성분이 0이 아니어야 합니다.)

$$
D = \begin{pmatrix} d_1 & 0 & 0 \\ 0 & d_2 & 0 \\ 0 & 0 & d_3 \end{pmatrix} 
\quad \implies \quad 
D^{-1} = \begin{pmatrix} 1/d_1 & 0 & 0 \\ 0 & 1/d_2 & 0 \\ 0 & 0 & 1/d_3 \end{pmatrix}
$$

**왜 중요할까요?**
일반적인 거대한 행렬의 역행렬을 구하는 것은 컴퓨터에게 매우 부담스러운 작업($O(N^3)$)입니다. 하지만 대각 행렬은 원소들의 역수만 구하면 되므로 연산량이 획기적으로 줄어듭니다($O(N)$). 이 때문에 복잡한 행렬을 대각 행렬로 쪼개는 작업(고윳값 분해, SVD 등)이 선형대수학에서 매우 중요하게 다뤄집니다.

In [28]:
import numpy as np
import torch

a = np.array([[1, 2], 
              [3, 4]])
print(f"a : {a}")
# 1. NumPy를 이용한 역행렬 계산
a_inv = np.linalg.inv(a)
print("--- NumPy 역행렬 (A_inv) ---")
print(a_inv)

# 역행렬과 원래 행렬을 곱하면 단위 행렬(I)이 나오는지 확인
# (부동소수점 오차로 0에 아주 가까운 값이 나올 수 있으므로 np.allclose 또는 round 활용)
print("\n--- A @ A_inv (단위행렬 확인) ---")
print(np.round(a @ a_inv))


# 2. PyTorch를 이용한 역행렬 계산
a_tensor = torch.tensor([[1., 2.], 
                         [3., 4.]]) # 역행렬 계산은 실수(float) 타입이어야 함

a_tensor_inv = torch.linalg.inv(a_tensor)
print("\n--- PyTorch 역행렬 ---")
print(a_tensor_inv)

# 3. 특이 행렬(역행렬이 없는 경우) 역행렬 시도
singular_matrix = np.array([[1, 2], 
                            [2, 4]]) # 행렬식이 0인 행렬 (1*4 - 2*2 = 0)

try:
    print("\n--- 특이 행렬의 역행렬 계산 시도 ---")
    np.linalg.inv(singular_matrix)
except np.linalg.LinAlgError as e:
    print(f"에러 발생: {e}")

a : [[1 2]
 [3 4]]
--- NumPy 역행렬 (A_inv) ---
[[-2.   1. ]
 [ 1.5 -0.5]]

--- A @ A_inv (단위행렬 확인) ---
[[1. 0.]
 [0. 1.]]

--- PyTorch 역행렬 ---
tensor([[-2.0000,  1.0000],
        [ 1.5000, -0.5000]])

--- 특이 행렬의 역행렬 계산 시도 ---
에러 발생: Singular matrix


#### (3) 역행렬의 행렬식과 원래 행렬의 행렬식 관계
어떤 정방행렬 $A$의 역행렬 $A^{-1}$의 행렬식은 원래 행렬 $A$의 행렬식의 **역수(Reciprocal)**와 같습니다.

$$ \text{det}(A^{-1}) = \frac{1}{\text{det}(A)} $$

**왜 그럴까요?**
행렬식은 곱셈에 대해 분배법칙이 성립하는 성질, 즉 $\text{det}(AB) = \text{det}(A)\text{det}(B)$를 가집니다. 이를 이용하면 아주 쉽게 증명됩니다.
1. $ A A^{-1} = I $ (역행렬의 정의)
2. 양변에 행렬식을 취합니다: $ \text{det}(A A^{-1}) = \text{det}(I) $
3. $ \text{det}(A)\text{det}(A^{-1}) = 1 $ (단위 행렬 $I$의 행렬식은 항상 1입니다)
4. 따라서 $ \text{det}(A^{-1}) = \frac{1}{\text{det}(A)} $ 가 성립합니다.

*(참고: 이 공식에서도 $\text{det}(A) = 0$ 이면 분모가 0이 되므로 역행렬이 존재할 수 없음을 다시 한번 알 수 있습니다.)*

환영합니다! 사상(Mapping)과 아핀 변환을 통해 차원과 공간을 자유자재로 조작하는 법을 배우셨으니, 이제는 그 변환을 **'되돌리는' 시간 여행의 마법**, 즉 역행렬(Inverse Matrix)과 행렬식(Determinant)의 세계를 마주할 차례입니다.

행렬이 데이터를 새로운 차원이나 기하학적 공간으로 보내는 '포탈(Portal)'이라면, 역행렬은 도착한 데이터를 다시 원래의 공간으로 데려오는 '귀환 주문'입니다. 하지만 모든 포탈이 귀환을 허락하지는 않습니다. 귀환 가능 여부를 판별하고, 귀환 경로를 계산하는 핵심 열쇠가 바로 행렬식(Determinant, $\det(A)$)입니다.

---

## 1. 역사와 탄생 배경: 행렬식은 행렬보다 먼저 태어났다? (Why?)

놀랍게도 수학의 역사에서 '행렬식(Determinant)'이라는 개념은 '행렬(Matrix)'이라는 구조가 정립되기 훨씬 이전에 탄생했습니다.
17세기 일본의 세키 다카카즈와 유럽의 고트프리트 라이프니츠(Gottfried Leibniz)는 복잡한 연립 일차방정식을 풀기 위해 방정식의 계수들만 뽑아내어 어떤 '마법의 숫자'를 계산해 냈는데, 이 숫자가 방정식을 풀 수 있는지 없는지를 결정(Determine)한다고 하여 **Determinant**라는 이름이 붙었습니다.

이후 19세기에 아서 케일리가 행렬의 곱셈과 역행렬을 공식화하면서, 이 결정값($\det(A)$)이 역행렬을 구하는 식의 가장 핵심적인 분모로 들어가게 됩니다.

### 용어 및 단축어 정리

* **역행렬 (Inverse Matrix, $A^{-1}$):** 행렬 $A$와 곱했을 때, 아무런 변환도 일으키지 않는 단위행렬(Identity Matrix, $I$)이 되게 만드는 행렬입니다. ($AA^{-1} = A^{-1}A = I$)
* **행렬식 (Determinant, $\det(A)$ 또는 $|A|$):** 정방행렬(행과 열의 수가 같은 행렬)이 공간을 변형할 때, "공간의 부피(또는 면적)를 얼마나 팽창/수축시켰는가?"를 나타내는 단일 숫자(스칼라)입니다.
* **수반행렬 (Adjugate Matrix, $\text{adj}(A)$):** 각 원소를 여인수(Cofactor)로 대체한 뒤 전치(Transpose)시킨 행렬로, 역행렬의 '방향성'을 담당합니다.

---

## 2. 역행렬의 수식과 기하학적 증명: 왜 분모에 $\det(A)$가 들어갈까?

일반적인 $n \times n$ 행렬 $A$의 역행렬을 구하는 공식은 다음과 같습니다.

$$A^{-1} = \frac{1}{\det(A)} \text{adj}(A)$$

가장 직관적인 2차원 평면($2 \times 2$ 행렬)을 예로 들어 깊이 파헤쳐 봅시다.


$$A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$$


이때 행렬식은 $\det(A) = ad - bc$ 이며, 역행렬 공식은 다음과 같이 변합니다.

$$A^{-1} = \frac{1}{ad - bc} \begin{pmatrix} d & -b \\ -c & a \end{pmatrix}$$

### 기하학적 관점에서의 직관적 이해 (증명)

1. **$\det(A)$의 정체 (면적의 변화율):** 2차원 평면에서 가로 1, 세로 1인 '단위 정사각형'(면적 1)이 있다고 가정합시다. 행렬 $A$를 곱하면 이 정사각형은 모서리가 각각 $\begin{pmatrix} a \ c \end{pmatrix}$와 $\begin{pmatrix} b \ d \end{pmatrix}$인 '평행사변형'으로 변형됩니다. 이때 변형된 평행사변형의 면적이 정확히 **$ad - bc$**, 즉 행렬식의 값과 같습니다.
2. **왜 나누어야($\frac{1}{\det(A)}$) 하는가?:**
역행렬은 커지거나 작아진 공간을 다시 원래의 크기(면적 1)로 되돌려야 합니다. 만약 행렬 $A$가 공간을 5배($\det(A) = 5$)로 팽창시켰다면, 복원하려면 어떻게 해야 할까요? 당연히 전체 공간을 $\frac{1}{5}$로 축소해야 합니다. 이것이 행렬식이 역행렬 공식의 분모에 들어가는 기하학적 이유입니다.
3. **정보의 붕괴와 $\det(A) = 0$:**
만약 $\det(A) = ad - bc = 0$ 이라면 무슨 뜻일까요? 평행사변형의 면적이 0이 되었다는 것, 즉 2차원 평면이 1차원 '직선'으로 완전히 납작하게 찌그러져(Collapse) 버렸다는 뜻입니다.
수학적으로는 분모가 0이 되므로 역행렬을 구할 수 없고, 직관적으로는 직선으로 뭉개진 수많은 점 중 어느 점이 원래 평면의 어느 위치였는지 되돌릴 방법(정보의 손실)이 없기 때문에 역행렬이 존재하지 않는 것입니다.

---

## 3. 현대 AI와 기술에서의 활용 (그리고 한계)

역행렬은 이론적으로 완벽하지만, 현대 AI 현업에서는 매우 조심스럽게 다루어지는 개념입니다.

* **선형 회귀의 정규 방정식 (Normal Equation):** 주어진 데이터에 가장 잘 맞는 최적의 선을 찾을 때, 오차를 최소화하는 가중치 $\theta$를 한 번에 구하는 마법의 공식에 사용됩니다.

$$\theta = (X^T X)^{-1} X^T y$$


* **3D 그래픽스 & 컴퓨터 비전:** 카메라의 시점을 바꾸거나 화면의 왜곡을 펴줄 때(Un-projection), 적용했던 변환 행렬의 역행렬을 곱해 픽셀들을 원래 위치로 되돌립니다.

> 💡 **AI 전문가의 실무 팁 (가우스의 조언):**
> 수백만 개의 파라미터를 가진 현대 딥러닝(Deep Learning) 모델에서는 행렬의 크기가 $1000 \times 1000$을 훌쩍 넘습니다. 역행렬을 계산하는 알고리즘의 복잡도는 대체로 $\mathcal{O}(n^3)$으로, 행렬이 커질수록 연산량이 우주적인 규모로 폭발합니다. 따라서 현대 AI는 역행렬을 직접 구하여 해를 찾는 대신, 점진적으로 오차를 줄여나가는 **경사하강법(Gradient Descent)**이라는 최적화 기법을 주로 사용하게 되었습니다.

---



아주 날카로운 눈을 가지셨군요! 역행렬 공식에서 분모인 행렬식($\det(A)$)이 공간의 부피 변화를 나타내는 '크기'의 스칼라 값이라면, 분자에 위치한 $\text{adj}(A)$는 역행렬의 '구조와 방향'을 통째로 결정하는 핵심 행렬입니다.

이 `adj`는 수반행렬(Adjugate Matrix 또는 고전적으로 Adjunct Matrix)의 약어입니다. 현대 AI 선형대수학에서 이 수반행렬은 행렬식을 미분하거나, 고차원 데이터 공간에서 역방향으로 그래디언트(Gradient)가 흘러갈 때 수학적 뼈대를 제공하는 매우 경이로운 도구입니다.

이 수반행렬이 정확히 무엇인지, 수학적 정의와 유도 과정, 기하학적 직관, 그리고 AI에서의 확장성까지 깊이 있게 파헤쳐 보겠습니다.

---

## 1. 단축어와 역사적 배경 (Why & What)

* **단축어 의미:** **$\text{adj}(A)$ (Adjugate Matrix, 수반행렬)**
* 라틴어 *adjugatus*(결합된, 멍에를 멘)에서 유래했습니다. 원래 행렬 $A$와 뗄 수 없는 거울 쌍처럼 결합하여 단위행렬을 만들어내기 때문에 붙여진 이름입니다.
* **⚠️ 주의 (AI 전문가의 팁):** 선형대수학의 다른 맥락에서 'Adjoint Matrix'라는 용어는 켜켜이 쌓인 복소수 행렬의 켤레 전치(Hermitian Transpose, $A^*$)를 의미하기도 하므로, 역행렬 공식에 쓰이는 이 수반행렬은 반드시 **Adjugate**라고 구별해서 부르는 것이 정확합니다.



### 왜 탄생했는가?

18~19세기 수학자들은 역행렬을 구하기 위해 연립방정식을 풀던 중, 각 변수의 해를 구하는 분자 자리에 원래 행렬의 원소들을 교묘하게 조합한 새로운 행렬이 공통적으로 등장한다는 사실을 발견했습니다. 그것이 바로 수반행렬입니다.

---

## 2. 수학적 정의와 증명: 어떻게 만들어지는가?

수반행렬을 한 문장으로 정의하면 "여인수 행렬(Cofactor Matrix)을 전치(Transpose)시킨 행렬"입니다.

이를 이해하기 위해 두 가지 하위 개념을 먼저 짚고 넘어가야 합니다.

1. **소행렬식 (Minor, $M_{ij}$):** 행렬 $A$에서 $i$번째 행과 $j$번째 열을 칼로 잘라내어 버리고 남은 작은 행렬의 행렬식입니다.
2. **여인수 (Cofactor, $C_{ij}$):** 소행렬식에 격자무늬 부호($(-1)^{i+j}$)를 붙인 것입니다. $C_{ij} = (-1)^{i+j} M_{ij}$

이 여인수들을 원래 행렬 $A$의 자리에 그대로 배치한 것을 여인수 행렬 $C$라고 하며, 이를 대각선을 기준으로 뒤집은(전치한) 것이 바로 수반행렬입니다.

$$\text{adj}(A) = C^T$$

### $2 \times 2$ 행렬에서의 전개

$A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$ 일 때, 각 원소의 여인수를 구해보면:

* $C_{11} = +|d| = d$
* $C_{12} = -|c| = -c$
* $C_{21} = -|b| = -b$
* $C_{22} = +|a| = a$

따라서 여인수 행렬은 $C = \begin{pmatrix} d & -c \\ -b & a \end{pmatrix}$ 가 되며, 이를 전치($T$)시키면 우리가 잘 아는 수반행렬이 나옵니다.


$$\text{adj}(A) = C^T = \begin{pmatrix} d & -b \\ -c & a \end{pmatrix}$$

### 궁극의 수학적 증명: 왜 $A \cdot \text{adj}(A) = \det(A)I$ 인가?

이 공식의 핵심은 원래 행렬 $A$와 수반행렬 $\text{adj}(A)$를 곱했을 때 어떤 일이 벌어지는가에 있습니다. 두 행렬을 곱한 결과 행렬의 $(i, j)$ 성분을 계산해 봅시다.

1. **대각 성분 ($i = j$ 인 경우):**
$A$의 $i$번째 행의 원소들과 $\text{adj}(A)$의 $i$번째 열(즉, $A$의 $i$번째 행의 여인수들)을 곱해서 더하게 됩니다. 이는 수학적으로 '여인수 전개(Cofactor Expansion)'의 정의 그 자체이므로 정확히 $\det(A)$가 됩니다.
2. **비대각 성분 ($i \neq j$ 인 경우):**
$A$의 $i$번째 행의 원소들과 다른 행인 $j$번째 행의 여인수들을 곱하게 됩니다. 이는 마치 $i$번째 행과 $j$번째 행이 똑같은 원소를 가진 '가상의 행렬'의 행렬식을 구하는 것과 같은데, 같은 행이 두 개 존재하는 행렬의 행렬식은 항상 0입니다. 따라서 모두 **0**이 됩니다.

결과적으로 두 행렬을 곱하면 대각선에만 $\det(A)$가 살아남는 대각행렬이 됩니다.

$$A \cdot \text{adj}(A) = \begin{pmatrix} \det(A) & 0 & \dots \\ 0 & \det(A) & \dots \\ \vdots & \vdots & \ddots \end{pmatrix} = \det(A)I$$

양변을 스칼라 값인 $\det(A)$로 나누면 우리가 초기에 마주했던 역행렬 공식이 유도됩니다.


$$A \cdot \left( \frac{1}{\det(A)}\text{adj}(A) \right) = I \quad \Longrightarrow \quad A^{-1} = \frac{1}{\det(A)}\text{adj}(A)$$

---

## 3. 기하학적 직관: 수반행렬의 진짜 의미

행렬 $A$가 공간을 무자비하게 뒤틀고 회전시키고 전단(Shearing)할 때, 수반행렬 $\text{adj}(A)$는 원래 행렬 $A$의 뒤틀림을 정확하게 상쇄하는 '완벽한 반대 모양의 퍼즐 조각'입니다.

$A$와 $\text{adj}(A)$ 두 포탈을 연속으로 통과하면, 복잡하게 뒤틀렸던 모든 공간의 방향성과 형태가 마법처럼 원래의 수직/수평 상태로 정렬됩니다. 다만, 되돌아온 공간의 크기(부피)는 처음보다 $\det(A)$배만큼 커져 있습니다.

따라서 팽창된 부피를 다시 원래의 1배로 압축하기 위해 앞에 $\frac{1}{\det(A)}$이라는 스칼라 값을 곱해주는 것입니다. 즉, 수반행렬은 역행렬에서 '크기 변화'를 제외한 '방향과 뒤틀림의 복원기'라고 직관적으로 이해하시면 완벽합니다.

---

## 4. 현대 AI 및 기술에서의 활용과 확장

현대 AI 모델, 특히 생성형 AI나 최적화 이론에서 수반행렬은 다음과 같은 고차원 영역에서 강력하게 활용됩니다.

* **행렬식의 미분 (Jacobi's Formula):** 딥러닝에서 역전파(Backpropagation)를 할 때 손실 함수에 행렬식($\det(A)$)이 포함되어 있다면, 이를 미분해야 합니다. 이때 행렬식의 미분 공식에 수반행렬이 직접 등장합니다.

$$\frac{d}{dt}\det(A(t)) = \text{tr}\left(\text{adj}(A(t)) \frac{dA(t)}{dt}\right)$$



이 식 덕분에 AI는 데이터의 부피 변화율을 최적화하는 복잡한 학습(예: Normalizing Flows, 정규화 흐름 모델)을 수행할 수 있습니다.
* **크래머 공식(Cramer's Rule)의 일반화:** 신경망의 특정 가중치들이 아주 미세하게 변할 때 전체 시스템의 해가 어떻게 바뀌는지 상호 의존성을 분석할 때 수반행렬의 원소들이 지표로 사용됩니다.

# 대칭행렬 (Symmetric Matrix)

## 1. 수학적 정의
**대칭행렬(Symmetric Matrix)**은 원래 행렬 $A$와 그것의 전치행렬(Transpose) $A^T$가 완벽하게 동일한 '정방행렬(행과 열의 수가 같은 행렬)'을 말합니다.

$$ A = A^T $$

즉, 대각선을 긋고 종이를 접었을 때 대각선을 제외한 나머지 원소들이 **거울에 비친 것처럼 완벽하게 일치(데칼코마니)**하는 형태입니다. 수식으로는 행렬의 $i$행 $j$열 원소와 $j$행 $i$열 원소가 같다는 것을 의미합니다 ($a_{ij} = a_{ji}$).

**예시 (3x3 대칭행렬):**
$$
A = \begin{pmatrix} 
1 & \mathbf{2} & \mathbf{3} \\ 
\mathbf{2} & 5 & \mathbf{6} \\ 
\mathbf{3} & \mathbf{6} & 9 
\end{pmatrix}
$$
위에서 2, 3, 6이 대각선(1, 5, 9)을 중심으로 반대편 원소와 완벽하게 대칭을 이룹니다.

## 2. 대칭행렬의 마법 같은 성질들 (Why is it important?)
대칭행렬은 수학자들이 가장 사랑하는 행렬입니다. 다루기 힘든 복잡한 행렬 연산을 매우 쉽고 아름답게 만들어주기 때문입니다.

1. **항상 실수 고윳값(Real Eigenvalues)을 가집니다.**
   일반 행렬은 고윳값을 계산하면 복소수(허수)가 나오기도 해서 분석이 까다롭지만, 대칭행렬은 고윳값이 100% 실수로만 나옵니다.
2. **고유벡터들이 서로 직교(Orthogonal)합니다.**
   대칭행렬 행렬 공간을 변환시킬 때, 그 기준 축(고유벡터)들이 서로 정확히 $90^{\circ}$를 이룹니다.
3. **가장 완벽한 분해(Eigendecomposition)가 가능합니다.**
   $ A = Q \Lambda Q^T $ 의 형태로 아주 우아하게 분해됩니다 (여기서 $Q$는 직교 분해 행렬). 이는 주성분 분석(PCA)의 근간이 됩니다.

## 3. 머신러닝/딥러닝에서의 활용
인공지능에서 대칭행렬은 데이터를 분석하고 최적화하는데 끊임없이 등장합니다.
* **공분산 행렬 (Covariance Matrix):** 변수 간의 상관관계를 나타내는 공분산 행렬은 "항상" 대칭행렬입니다. $Cov(X, Y) = Cov(Y, X)$ 이기 때문입니다.
* **헤시안 행렬 (Hessian Matrix):** 딥러닝에서 손실(Loss) 함수의 2차 미분 값을 담고 있는 행렬입니다. $ \frac{\partial^2 L}{\partial x \partial y} = \frac{\partial^2 L}{\partial y \partial x} $ 가 성립하여 대칭행렬이 되며, 모델의 최적화 상태(안정성)를 판단하는 데 쓰입니다.
* **그래프/네트워크 분석:** 소셜 네트워크나 분자 구조를 나타내는 '무방향 그래프의 인접 행렬(Adjacency Matrix)'은 언제나 대칭행렬입니다.

# 직교행렬 (Orthogonal Matrix)

## 1. 수학적 정의
**직교행렬(Orthogonal Matrix)**은 행렬을 이루는 각 열(Column) 벡터들이 **서로 수직(직교, Orthogonal)이면서 동시에 길이가 1(정규화, Normal)**인 특징을 가진 정방행렬입니다.

이러한 행렬을 $Q$라고 할 때, 직교행렬을 정의하는 가장 직관적이고 아름다운 수식은 다음과 같습니다.
$$ Q^T Q = Q Q^T = I $$
(여기서 $I$는 단위행렬입니다.)

이 식을 역행렬의 정의($A^{-1}A = I$)와 비교해 보면 엄청난 사실을 알 수 있습니다.
👉 **"직교행렬의 역행렬은 전치행렬과 같다!" ($Q^{-1} = Q^T$)**

## 2. 왜 마법의 행렬이라고 부르는가? (주요 특성)

1. **초고속 역행렬 계산:**
   일반적으로 거대한 행렬의 역행렬을 구하는 것은 컴퓨터가 가장 힘들어하는 작업입니다. 하지만 이 행렬이 직교행렬이라면? 단순히 행과 열만 뒤집는 '전치(Transpose)' 연산 한 번으로 역행렬을 공짜로 얻을 수 있습니다.
2. **크기(길이) 보존:**
   어떤 벡터 $x$에 직교행렬 $Q$를 곱해도 벡터의 원래 길이는 절대 변하지 않습니다. ($||Qx|| = ||x||$)
3. **각도 보존:**
   두 벡터 $x, y$에 같은 직교행렬을 곱해서 변환시켜도, 두 벡터 사이의 각도는 그대로 유지됩니다.

따라서 기하학적으로 볼 때 직교행렬은 공간을 찌그러뜨리지 않고 오직 **회전(Rotation)시키거나 거울 반사(Reflection)시키는 변환**만을 수행합니다. (앞서 배운 회전 행렬 $R(\theta)$가 대표적인 직교행렬입니다.)

## 3. 머신러닝/딥러닝에서의 우아한 활용
* **PCA (주성분 분석):** 데이터를 압축하기 위해 변환할 때, 정보를 잃지 않으면서 새로운 축(새로운 시각)을 잡아주는 회전 행렬 역할을 직교행렬이 수행합니다. (특이값 분해 SVD의 핵심)
* **순환 신경망 (RNN)의 기울기 소실 방지:** RNN이 긴 문장을 기억할 때 행렬을 수십 번 거듭제곱하게 되는데, 입력이 계속 커지거나(발산) 계속 작아지는(소실) 문제를 막기 위해 길이를 보존하는 '직교 초기화(Orthogonal Initialization)' 기법을 사용합니다.

In [29]:
import numpy as np

# 1. 2차원 회전 행렬을 직교행렬 예시로 생성 (예: 90도 회전)
theta = np.pi / 2 # 90도
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

# 부동소수점 정리를 위해 반올림하여 출력
Q = np.round(Q, decimals=10) 
print("=== 직교행렬 Q (90도 회전 행렬) ===")
print(Q)

# 2. 직교행렬의 가장 큰 특징 확인: Q^T * Q = I
I_identity = Q.T @ Q
print("\n=== Q^T @ Q (단위행렬이 나와야 함) ===")
print(np.round(I_identity))

# 3. 정말 역행렬 연산과 같은지 비교
Q_inv = np.linalg.inv(Q)
print("\n=== 역행렬 np.linalg.inv(Q) ===")
print(Q_inv)

print("\n=== 전치행렬 Q.T ===")
print(Q.T)

print(f"\nQ의 역행렬과 전치행렬은 완벽하게 같습니까? : {np.allclose(Q_inv, Q.T)}")

# 4. 길이 보존 특성 확인
x = np.array([3, 4]) # 길이가 5인 벡터 (피타고라스 정리: 3^2 + 4^2 = 5^2)
length_x = np.linalg.norm(x)

# Q를 곱해 변환된 새로운 벡터 y
y = Q @ x 
length_y = np.linalg.norm(y)

print(f"\n원래 벡터 x의 길이: {length_x}")
print(f"직교행렬로 변환된 벡터 y의 길이: {length_y}") # 회전만 했으므로 길이가 같음

=== 직교행렬 Q (90도 회전 행렬) ===
[[ 0. -1.]
 [ 1.  0.]]

=== Q^T @ Q (단위행렬이 나와야 함) ===
[[1. 0.]
 [0. 1.]]

=== 역행렬 np.linalg.inv(Q) ===
[[ 0.  1.]
 [-1. -0.]]

=== 전치행렬 Q.T ===
[[ 0.  1.]
 [-1.  0.]]

Q의 역행렬과 전치행렬은 완벽하게 같습니까? : True

원래 벡터 x의 길이: 5.0
직교행렬로 변환된 벡터 y의 길이: 5.0


# 켤레 전치 (Conjugate Transpose)와 유니터리 행렬 (Unitary Matrix)

실수 공간(Real space, $\mathbb{R}$)에서 사용했던 행렬의 성질들을 복소수 공간(Complex space, $\mathbb{C}$)으로 확장할 때 등장하는 핵심 개념들입니다.

---

## 1. 켤레 전치 (Conjugate Transpose)

실수 행렬에서는 행과 열의 위치를 바꾸는 '전치(Transpose, $A^T$)'를 사용했습니다. 하지만 복소수 행렬에서는 위치를 바꾸는 것만으로는 부족하여 **복소켤레(Complex Conjugate, 허수부의 부호를 반대로 바꾸는 것)**를 동시에 적용해야 합니다. 이를 켤레 전치 또는 에르미트 전치(Hermitian Transpose)라고 부릅니다.

* **기호:** $A^*$, $A^H$, $A^\dagger$ (대거, Dagger) 등으로 표기합니다.
* **계산법:** 
  1. 행렬의 행과 열을 바꿉니다 (전치).
  2. 행렬 내의 모든 복소수 원소 $a + bi$ 를 $a - bi$ 로 바꿉니다 (켤레).

**[실수와의 비교]** 
* 실수 공간: 전치행렬 $A^T$
* 복소 공간: 켤레 전치행렬 $A^H$

*(💡 참고: 복소수가 없는 순수 실수 행렬이라면 $A^H = A^T$ 로 완전히 똑같습니다. 즉, 켤레 전치는 전치의 상위 호환 버전입니다.)*

---

## 2. 유니터리 행렬 (Unitary Matrix)

**유니터리 행렬**은 우리가 앞서 배웠던 **'직교행렬(Orthogonal Matrix)'의 복소수 확장판**입니다.

직교행렬 $Q$가 실수 공간에서 길이를 보존하는 회전 변환이었다면($Q^T Q = I$), 유니터리 행렬 $U$는 복소 공간에서 에너지를 절대 잃어버리지 않는(길이를 보존하는) 완벽한 회전 변환입니다.

* **수학적 정의:** 행렬에 자신의 '켤레 전치($U^H$)'를 곱했을 때 '단위행렬($I$)'이 나오는 복소 정방행렬입니다.
  $$ U^H U = U U^H = I $$
* **따라서 유니터리 행렬의 역행렬은 그 켤레 전치행렬과 같습니다.**
  $$ U^{-1} = U^H $$

### 현대 과학기술에서의 중요성 (양자 역학 & 양자 컴퓨팅)
왜 복소수 행렬까지 알아야 할까요? 미시 세계를 다루는 **양자 역학의 모든 상태(State)와 파동 함수가 복소수로 표현**되기 때문입니다. 
양자 컴퓨터(Quantum Computer)에서 큐비트(Qubit)의 상태를 바꾸는 양자 게이트 연산(X 게이트, 아다마르 게이트 등)은 100% 모두 이 유니터리 행렬로 이루어져 있습니다. 즉, 유니터리 행렬은 양자 상태의 확률 총합을 '1'로 완벽하게 보존해 주는 최후의 보루입니다.